In [1]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb contractions spacy tqdm
print("Done")

In [2]:
import os, math, re, random, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from contractions import fix

from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, get_cosine_with_hard_restarts_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


Using device: cuda


In [3]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()

    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available

    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)

    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/content/15_labels_data.csv")  # Adjust file path if needed
df.rename(columns={"label": "labels"}, inplace=True)   # <- if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']



spaCy is using GPU: True


Preprocessing texts:   0%|          | 0/8417 [00:00<?, ?it/s]

In [4]:
# Ensure the preprocessed texts are attached back to df before converting:
df["text"] = processed_texts
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

# Infer number of classes
if "labels" not in train_dataset.column_names:
    raise ValueError("Column 'labels' not found. Rename your label column to 'labels' before continuing.")
num_labels = int(len(set(train_dataset["labels"])))
print("Detected num_labels:", num_labels)

Detected num_labels: 15


In [5]:
ENCODER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
text_encoder = AutoModel.from_pretrained(ENCODER_NAME)
# Freeze encoder (we only train diffusion heads)
for p in text_encoder.parameters():
    p.requires_grad = False
text_encoder.eval().to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [6]:
# ----------------
# 3) Tokenize datasets
# ----------------
def tok_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

train_enc = train_dataset.map(tok_fn, batched=True, remove_columns=[c for c in train_dataset.column_names if c not in ["text","labels"]])
eval_enc  = eval_dataset.map(tok_fn,  batched=True, remove_columns=[c for c in eval_dataset.column_names  if c not in ["text","labels"]])

train_enc.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
eval_enc.set_format(type="torch",  columns=["input_ids","attention_mask","labels"])


Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [7]:
# ----------------
# 4) Diffusion utilities (DDPM schedule)
# ----------------
class DiffusionSchedule:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)  # \bar{alpha}_t

    def to(self, device):
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alpha_bars = self.alpha_bars.to(device)
        return self

SCHED_T = 1000
sched = DiffusionSchedule(timesteps=SCHED_T).to(device)

In [8]:
# ----------------
# 5) Sinusoidal timestep embedding
# ----------------
def sinusoidal_time_embedding(t, dim):
    """
    t: (B,) integer timesteps
    returns (B, dim) float embeddings
    """
    half = dim // 2
    freqs = torch.exp(torch.arange(half, device=t.device) * -(math.log(10000) / (half - 1 + 1e-9)))
    # shape: (B, half)
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:  # pad if odd
        emb = torch.nn.functional.pad(emb, (0,1))
    return emb

In [9]:
# ----------------
# 6) Denoiser + Classifier
# ----------------
class DenoiserMLP(nn.Module):
    """
    Predicts noise eps_t from x_t and time embedding.
    Also returns a hidden feature vector for classification.
    """
    def __init__(self, latent_dim=768, time_dim=128, hidden_dim=1024):
        super().__init__()
        self.time_proj = nn.Sequential(
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU()
        )
        self.in_proj = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.SiLU()
        )
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        self.out_eps = nn.Linear(hidden_dim, latent_dim)  # predict epsilon
        self.feature_dim = hidden_dim

    def forward(self, x_t, t_emb):
        # x_t: (B, D), t_emb: (B, T)
        h = self.in_proj(x_t) + self.time_proj(t_emb)
        h = self.net(h)
        eps_pred = self.out_eps(h)
        return eps_pred, h  # h used for classification head

class ClassifierHead(nn.Module):
    def __init__(self, in_dim, num_labels):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, num_labels)
        )
    def forward(self, features):
        return self.head(features)

latent_dim = text_encoder.config.hidden_size  # 768 for BERT-base
time_emb_dim = 128
denoiser = DenoiserMLP(latent_dim=latent_dim, time_dim=time_emb_dim, hidden_dim=1024).to(device)
clf_head = ClassifierHead(denoiser.feature_dim, num_labels).to(device)


In [10]:
# ----------------
# 7) Helpers: get sentence embeddings
# ----------------
@torch.no_grad()
def sentence_embedding(input_ids, attention_mask):
    """
    Use frozen BERT CLS as x0. You can swap to mean-pooling if you prefer.
    """
    outputs = text_encoder(input_ids=input_ids, attention_mask=attention_mask)
    # CLS token: outputs.last_hidden_state[:,0,:]
    x0 = outputs.last_hidden_state[:, 0, :]  # (B, 768)
    return x0

In [11]:
# ----------------
# 8) DataLoaders
# ----------------
BATCH_SIZE = 32
train_loader = DataLoader(train_enc, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
eval_loader  = DataLoader(eval_enc,  batch_size=BATCH_SIZE, shuffle=False)

In [12]:
# ----------------
# 9) Training setup
# ----------------
EPOCHS = 5
LR = 3e-4
LAMBDA_NOISE = 0.8  # weight for denoising loss vs. classification
warmup_steps = 200
total_steps = EPOCHS * len(train_loader)

params = list(denoiser.parameters()) + list(clf_head.parameters())
optimizer = optim.AdamW(params, lr=LR, weight_decay=0.01)
scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps, num_cycles=1
)

mse = nn.MSELoss()
ce  = nn.CrossEntropyLoss()

In [13]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [15]:
def validate_batch(logits, labels):
    # Basic shapes/dtypes/devices
    assert logits.dim() == 2, f"logits must be [B, C], got {tuple(logits.shape)}"
    B, C = logits.shape
    assert torch.is_floating_point(logits), f"logits dtype must be float, got {logits.dtype}"
    assert logits.device.type == "cuda", "Expecting CUDA (just to be consistent while debugging)"

    # Bring labels to same device
    labels = labels.to(logits.device)

    # Squeeze accidental trailing dim: [B,1] -> [B]
    if labels.ndim == 2 and labels.shape[1] == 1:
        labels = labels.squeeze(1)

    # If labels are float one-hots/probabilities, handle differently (skip below checks)
    if labels.ndim == 2 and labels.shape[1] == C:
        # Soft labels case. Check row sums ~ 1.0
        row_sums = labels.sum(dim=1)
        bad = (~torch.isfinite(labels)).any() or (row_sums.isnan().any()) \
              or (row_sums < 0.99).any() or (row_sums > 1.01).any()
        assert not bad, "labels look like soft labels but rows don't sum to ~1 or contain NaNs/Infs"
        return labels  # Return as-is for soft-label CE

    # Otherwise we expect class indices
    if labels.dtype != torch.long:
        # If they were floats that should be integers, check they are all near integers.
        if torch.is_floating_point(labels):
            near_int = torch.allclose(labels, labels.round(), atol=1e-4, rtol=0)
            assert near_int, "labels are float and not integer-like; convert your pipeline to produce class IDs"
            labels = labels.round().long()
        else:
            labels = labels.long()

    # Range checks
    assert labels.ndim == 1 and labels.shape[0] == B, f"labels must be [B], got {tuple(labels.shape)}"
    assert torch.isfinite(logits).all(), "logits contain NaN/Inf"
    assert torch.isfinite(labels).all(), "labels contain NaN/Inf"

    invalid = (labels < 0) | (labels >= C)
    if invalid.any():
        bad_idx = torch.nonzero(invalid, as_tuple=False).squeeze(1)
        # Print a few offending examples
        print("Invalid labels at positions (up to 10):", bad_idx[:10].tolist())
        print("Offending label values (up to 10):", labels[bad_idx[:10]].tolist())
        print("C (num classes):", C)
        raise ValueError("Found out-of-range class indices")

    return labels




In [ ]:
# ----------------
# 10) Train loop
# ----------------
best_val_f1 = 0.0

def accuracy(pred, y):
    return (pred == y).float().mean().item()

def eval_epoch():
    denoiser.eval(); clf_head.eval()
    from sklearn.metrics import f1_score
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in eval_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            x0 = sentence_embedding(input_ids, attention_mask)  # (B, D)

            # For validation, we can skip noising and just classify using features at a mid timestep
            # (Averaging multiple t’s often helps; here we use one mid t for speed.)
            t = torch.full((x0.size(0),), int(0.6 * SCHED_T), device=device, dtype=torch.long)
            alpha_bar = sched.alpha_bars[t].unsqueeze(1)  # (B,1)
            noise = torch.zeros_like(x0)  # no added noise at eval feature extraction
            x_t = torch.sqrt(alpha_bar) * x0 + torch.sqrt(1 - alpha_bar) * noise

            t_emb = sinusoidal_time_embedding(t, time_emb_dim)
            _, h = denoiser(x_t, t_emb)
            logits = clf_head(h)
            preds = torch.argmax(logits, dim=-1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    f1 = f1_score(all_labels, all_preds, average="macro")
    acc = (all_preds == all_labels).mean()
    return f1, acc

for epoch in range(1, EPOCHS + 1):
    denoiser.train(); clf_head.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # 1) Get clean embeddings x0
        with torch.no_grad():
            x0 = sentence_embedding(input_ids, attention_mask)  # (B, D)

        # 2) Sample a random timestep t for each sample
        t = torch.randint(low=1, high=SCHED_T, size=(x0.size(0),), device=device)  # (B,)
        alpha_bar = sched.alpha_bars[t].unsqueeze(1)  # (B,1)

        # 3) Add noise
        eps = torch.randn_like(x0)
        x_t = torch.sqrt(alpha_bar) * x0 + torch.sqrt(1 - alpha_bar) * eps

        # 4) Denoiser predicts epsilon (noise)
        t_emb = sinusoidal_time_embedding(t, time_emb_dim)
        eps_pred, features = denoiser(x_t, t_emb)

        # 5) Classification from features
        logits = clf_head(features)

        # 6) Losses
        loss_noise = mse(eps_pred, eps)

        labels = labels.to(logits.device)
        if labels.ndim == 2 and labels.shape[1] == 1:
            labels = labels.squeeze(1)
        labels = labels.long()

        loss_ce = ce(logits, labels)

        loss = LAMBDA_NOISE * loss_noise + (1.0 - LAMBDA_NOISE) * loss_ce

        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            preds = torch.argmax(logits, dim=-1)
            acc_b = (preds == labels).float().mean().item()

        pbar.set_postfix(loss=float(loss.item()), ce=float(loss_ce.item()), mse=float(loss_noise.item()), acc=acc_b)

    # ---- Eval ----
    val_f1, val_acc = eval_epoch()
    print(f"Epoch {epoch}: val_f1={val_f1:.4f} | val_acc={val_acc:.4f}")

# ----------------
# 11) Inference helper
# ----------------
@torch.no_grad()
def predict_texts(texts, t_ratio=0.6):
    model_inputs = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
    x0 = sentence_embedding(model_inputs["input_ids"], model_inputs["attention_mask"])
    t = torch.full((x0.size(0),), int(t_ratio * SCHED_T), device=device, dtype=torch.long)
    alpha_bar = sched.alpha_bars[t].unsqueeze(1)
    x_t = torch.sqrt(alpha_bar) * x0  # no noise during deterministic eval
    t_emb = sinusoidal_time_embedding(t, time_emb_dim)
    _, h = denoiser(x_t, t_emb)
    logits = clf_head(h)
    return torch.softmax(logits, dim=-1).cpu().numpy(), torch.argmax(logits, dim=-1).cpu().numpy()

# Example:
# probs, preds = predict_texts(["This contract is voidable due to fraud.", "Patent infringement claim."])
# print(preds)

Epoch 1/5:   0%|          | 0/236 [00:00<?, ?it/s]

Epoch 1: val_f1=0.0241 | val_acc=0.2031


Epoch 2/5:   0%|          | 0/236 [00:00<?, ?it/s]